In [1]:
import os, random, shutil, json, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
from PIL import Image
from collections import defaultdict
from IPython.display import FileLink, display
import kagglehub, yaml

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
!pip install efficientnet_pytorch --quiet --no-deps
!pip install roboflow --quiet
from efficientnet_pytorch import EfficientNet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 6.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 85.1 MB/s eta 0:00:00:00:01


In [3]:
# PlantVillage
pv_path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
PLANTVILLAGE_DIR = os.path.join(pv_path, 'color')

# PlantWild
from huggingface_hub import snapshot_download
import zipfile
pw_snapshot = snapshot_download(repo_id="uqtwei2/PlantWild", repo_type="dataset")
pw_extract_dir = "/kaggle/working/plantwild_extracted"
if not os.path.isdir(pw_extract_dir):
    os.makedirs(pw_extract_dir, exist_ok=True)
    with zipfile.ZipFile(os.path.join(pw_snapshot, "plantwild.zip"), 'r') as z:
        z.extractall(pw_extract_dir)
PLANTWILD_DIR = os.path.join(pw_extract_dir, 'plantwild', 'images')

# PlantDoc
plantdoc_path = kagglehub.dataset_download("nirmalsankalana/plantdoc-dataset")
def find_plantdoc_root(search_from):
    for root, dirs, files in os.walk(search_from):
        if 'train' in dirs and 'test' in dirs:
            return root
    return None
plantdoc_dataset_root = find_plantdoc_root(plantdoc_path)

# FieldPlant
fieldplant_root = '/kaggle/working/FieldPlant-11'
if not os.path.isdir(fieldplant_root):
    from roboflow import Roboflow
    rf = Roboflow(api_key="U4BSNMZOjjYjIQlB6UkB")
    project = rf.workspace("plant-disease-detection").project("fieldplant")
    version = project.version(11)
    fp_dataset = version.download("yolov8")
    fieldplant_root = fp_dataset.location

# PaddyDoctor - NOTE: you must click "Join Competition" on
# kaggle.com/competitions/paddy-disease-classification once, in your browser, first
!kaggle competitions download -c paddy-disease-classification -p /kaggle/working/paddydoctor
paddydoctor_path = '/kaggle/working/paddydoctor'
if not os.path.isdir(os.path.join(paddydoctor_path, 'train_images')):
    with zipfile.ZipFile(os.path.join(paddydoctor_path, 'paddy-disease-classification.zip'), 'r') as z:
        z.extractall(paddydoctor_path)

# DhanShomadhan
dhanshomadhan_path = kagglehub.dataset_download("nirmalsankalana/dhan-shomadhan")

# WheatLong
wheatlong_path = kagglehub.dataset_download("yasserhessein/wheat-disease-dataset-small")

print("PlantVillage:", PLANTVILLAGE_DIR)
print("PlantWild:", PLANTWILD_DIR)
print("PlantDoc:", plantdoc_dataset_root)
print("FieldPlant:", fieldplant_root)
print("PaddyDoctor:", paddydoctor_path)
print("DhanShomadhan:", dhanshomadhan_path)
print("WheatLong:", wheatlong_path)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

loading Roboflow workspace...
loading Roboflow project...




Extracting Dataset Version Zip to FieldPlant-11 in yolov8::   0%|          | 0/10318 [00:00<?, ?it/s]
Extracting Dataset Version Zip to FieldPlant-11 in yolov8::   0%|          | 51/10318 [00:00<00:20, 507.94it/s]
Extracting Dataset Version Zip to FieldPlant-11 in yolov8::   1%|          | 102/10318 [00:00<00:22, 453.18it/s]
Extracting Dataset Version Zip to FieldPlant-11 in yolov8::   1%|▏         | 154/10318 [00:00<00:21, 480.91it/s]
Extracting Dataset Version Zip to FieldPlant-11 in yolov8::   2%|▏         | 212/10318 [00:00<00:19, 514.65it/s]
Extracting Dataset Version Zip to FieldPlant-11 in yolov8::   3%|▎         | 272/10318 [00:00<00:18, 543.16it/s]
Extracting Dataset Version Zip to FieldPlant-11 in yolov8::   3%|▎         | 335/10318 [00:00<00:17, 571.60it/s]
Extracting Dataset Version Zip to FieldPlant-11 in yolov8::   4%|▍         | 403/10318 [00:00<00:16, 605.77it/s]
Extracting Dataset Version Zip to FieldPlant-11 in yolov8::   5%|▍         | 476/10318 [00:00<00:15, 643.7

100%|███████████████████████████████████████| 1.02G/1.02G [00:06<00:00, 170MB/s]

PlantVillage: /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color
PlantWild: /kaggle/working/plantwild_extracted/plantwild/images
PlantDoc: /kaggle/input/datasets/nirmalsankalana/plantdoc-dataset
FieldPlant: /kaggle/working/FieldPlant-11
PaddyDoctor: /kaggle/working/paddydoctor
DhanShomadhan: /kaggle/input/datasets/nirmalsankalana/dhan-shomadhan
WheatLong: /kaggle/input/datasets/yasserhessein/wheat-disease-dataset-small


In [4]:
PLANTVILLAGE_CLASS_MAP = {
    'Potato___Early_blight': 'Potato_Early_Blight', 'Potato___Late_blight': 'Potato_Late_Blight',
    'Potato___healthy': 'Potato_Healthy', 'Tomato___Bacterial_spot': 'Tomato_Bacterial_Spot',
    'Tomato___Early_blight': 'Tomato_Early_Blight', 'Tomato___Late_blight': 'Tomato_Late_Blight',
    'Tomato___Leaf_Mold': 'Tomato_Leaf_Mold', 'Tomato___Septoria_leaf_spot': 'Tomato_Septoria_Leaf_Spot',
    'Tomato___Spider_mites Two-spotted_spider_mite': 'Tomato_Spider_Mites',
    'Tomato___Target_Spot': 'Tomato_Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus': 'Tomato_Mosaic_Virus', 'Tomato___healthy': 'Tomato_Healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Maize_Gray_Leaf_Spot',
    'Corn_(maize)___Common_rust_': 'Maize_Common_Rust',
    'Corn_(maize)___Northern_Leaf_Blight': 'Maize_Northern_Leaf_Blight',
    'Corn_(maize)___healthy': 'Maize_Healthy',
}

PLANTWILD_CLASS_MAP = {
    'corn gray leaf spot': 'Maize_Gray_Leaf_Spot', 'corn rust': 'Maize_Common_Rust',
    'corn northern leaf blight': 'Maize_Northern_Leaf_Blight', 'corn leaf': 'Maize_Healthy',
    'potato early blight': 'Potato_Early_Blight', 'potato late blight': 'Potato_Late_Blight',
    'potato leaf': 'Potato_Healthy', 'tomato early blight': 'Tomato_Early_Blight',
    'tomato late blight': 'Tomato_Late_Blight', 'tomato leaf mold': 'Tomato_Leaf_Mold',
    'tomato septoria leaf spot': 'Tomato_Septoria_Leaf_Spot',
    'tomato yellow leaf curl virus': 'Tomato_Yellow_Leaf_Curl_Virus',
    'tomato bacterial leaf spot': 'Tomato_Bacterial_Spot', 'tomato mosaic virus': 'Tomato_Mosaic_Virus',
    'tomato leaf': 'Tomato_Healthy',
}

PLANTDOC_CLASS_MAP = {
    'Corn_rust_leaf': 'Maize_Common_Rust', 'Corn_Gray_leaf_spot': 'Maize_Gray_Leaf_Spot',
    'Corn_leaf_blight': 'Maize_Northern_Leaf_Blight', 'Potato_leaf_early_blight': 'Potato_Early_Blight',
    'Potato_leaf_late_blight': 'Potato_Late_Blight', 'Tomato_leaf_bacterial_spot': 'Tomato_Bacterial_Spot',
    'Tomato_Early_blight_leaf': 'Tomato_Early_Blight', 'Tomato_leaf': 'Tomato_Healthy',
    'Tomato_leaf_late_blight': 'Tomato_Late_Blight', 'Tomato_mold_leaf': 'Tomato_Leaf_Mold',
    'Tomato_leaf_mosaic_virus': 'Tomato_Mosaic_Virus', 'Tomato_Septoria_leaf_spot': 'Tomato_Septoria_Leaf_Spot',
    'Tomato_two_spotted_spider_mites_leaf': 'Tomato_Spider_Mites',
    'Tomato_leaf_yellow_virus': 'Tomato_Yellow_Leaf_Curl_Virus',
}

FIELDPLANT_CLASS_MAP = {
    'Corn Gray leaf spot': 'Maize_Gray_Leaf_Spot', 'Corn Healthy': 'Maize_Healthy',
    'Corn rust leaf': 'Maize_Common_Rust',
    'Tomato healthy': 'Tomato_Healthy', 'Tomato leaf mosaic virus': 'Tomato_Mosaic_Virus',
    'Tomato leaf yellow virus': 'Tomato_Yellow_Leaf_Curl_Virus',
}

PADDYDOCTOR_CLASS_MAP = {
    'bacterial_leaf_blight': 'Rice_Bacterial_Leaf_Blight', 'bacterial_leaf_streak': 'Rice_Bacterial_Leaf_Streak',
    'bacterial_panicle_blight': 'Rice_Bacterial_Panicle_Blight', 'blast': 'Rice_Blast',
    'brown_spot': 'Rice_Brown_Spot', 'dead_heart': 'Rice_Dead_Heart', 'downy_mildew': 'Rice_Downy_Mildew',
    'hispa': 'Rice_Hispa', 'tungro': 'Rice_Tungro', 'normal': 'Rice_Healthy',
}

DHANSHOMADHAN_CLASS_MAP = {
    'Rice___blast': 'Rice_Blast', 'Rice___tungro': 'Rice_Tungro', 'Rice___brown_spot': 'Rice_Brown_Spot',
    'Rice___leaf_scald': 'Rice_Leaf_Scald',
    'Rice___bacterial_blight': 'Rice_Sheath_Blight',  # corrected per paper's official disease list
}

WHEATLONG_CLASS_MAP = {
    'Mildew': 'Wheat_Mildew', 'YellowRust': 'Wheat_Yellow_Rust', 'Septoria': 'Wheat_Septoria',
    'Healthy': 'Wheat_Healthy', 'BrownRust': 'Wheat_Brown_Rust',
}

CLASS_NAMES = [
    'Maize_Common_Rust', 'Maize_Gray_Leaf_Spot', 'Maize_Healthy', 'Maize_Northern_Leaf_Blight',
    'Potato_Early_Blight', 'Potato_Healthy', 'Potato_Late_Blight',
    'Rice_Bacterial_Leaf_Blight', 'Rice_Bacterial_Leaf_Streak', 'Rice_Bacterial_Panicle_Blight',
    'Rice_Blast', 'Rice_Brown_Spot', 'Rice_Dead_Heart', 'Rice_Downy_Mildew',
    'Rice_Healthy', 'Rice_Hispa', 'Rice_Leaf_Scald', 'Rice_Sheath_Blight', 'Rice_Tungro',
    'Tomato_Bacterial_Spot', 'Tomato_Early_Blight', 'Tomato_Healthy', 'Tomato_Late_Blight',
    'Tomato_Leaf_Mold', 'Tomato_Mosaic_Virus', 'Tomato_Septoria_Leaf_Spot',
    'Tomato_Spider_Mites', 'Tomato_Target_Spot', 'Tomato_Yellow_Leaf_Curl_Virus',
    'Wheat_Brown_Rust', 'Wheat_Healthy', 'Wheat_Mildew', 'Wheat_Septoria', 'Wheat_Yellow_Rust',
]
print(f"Total classes: {len(CLASS_NAMES)}")

Total classes: 34


In [5]:
def gather_source_images(source_dir, class_map):
    bucket = defaultdict(list)
    for source_class, unified_class in class_map.items():
        class_dir = os.path.join(source_dir, source_class)
        if not os.path.isdir(class_dir):
            print(f"  WARNING: not found: {class_dir}")
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                bucket[unified_class].append(os.path.join(class_dir, fname))
    return bucket

def gather_plantdoc_train_images(plantdoc_root, class_map):
    bucket = defaultdict(list)
    train_dir = os.path.join(plantdoc_root, 'train')
    for source_class, unified_class in class_map.items():
        class_dir = os.path.join(train_dir, source_class)
        if not os.path.isdir(class_dir):
            print(f"  WARNING: not found: {class_dir}")
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                bucket[unified_class].append(os.path.join(class_dir, fname))
    return bucket

def gather_fieldplant_images(root, class_map):
    with open(os.path.join(root, 'data.yaml')) as f:
        data_yaml = yaml.safe_load(f)
    class_id_to_name = {i: name for i, name in enumerate(data_yaml['names'])}
    bucket = defaultdict(list)
    for split in ['train', 'valid', 'test']:
        images_dir = os.path.join(root, split, 'images')
        labels_dir = os.path.join(root, split, 'labels')
        if not os.path.isdir(images_dir):
            continue
        for label_file in os.listdir(labels_dir):
            if not label_file.endswith('.txt'):
                continue
            with open(os.path.join(labels_dir, label_file)) as f:
                lines = f.readlines()
            class_ids_in_image = {int(l.strip().split()[0]) for l in lines if l.strip()}
            source_names = {class_id_to_name[cid] for cid in class_ids_in_image}
            mapped_names = {n for n in source_names if n in class_map}
            if not mapped_names or len(source_names) > len(mapped_names) or len(mapped_names) > 1:
                continue
            unified_class = class_map[list(mapped_names)[0]]
            img_path = os.path.join(images_dir, label_file.replace('.txt', '.jpg'))
            if os.path.isfile(img_path):
                bucket[unified_class].append(img_path)
    return bucket

pv_bucket = gather_source_images(PLANTVILLAGE_DIR, PLANTVILLAGE_CLASS_MAP)
pw_bucket = gather_source_images(PLANTWILD_DIR, PLANTWILD_CLASS_MAP)
pd_bucket = gather_plantdoc_train_images(plantdoc_dataset_root, PLANTDOC_CLASS_MAP)
fp_bucket = gather_fieldplant_images(fieldplant_root, FIELDPLANT_CLASS_MAP)

paddydoctor_train_root = os.path.join(paddydoctor_path, 'train_images')
pdoc_bucket = gather_source_images(paddydoctor_train_root, PADDYDOCTOR_CLASS_MAP)
dhan_bucket = gather_source_images(dhanshomadhan_path, DHANSHOMADHAN_CLASS_MAP)

wheatlong_root = os.path.join(wheatlong_path, 'Wheat Disease Dataset')
wheat_bucket = gather_source_images(wheatlong_root, WHEATLONG_CLASS_MAP)

rice_merged = defaultdict(list)
for bucket in (pdoc_bucket, dhan_bucket):
    for cls, paths in bucket.items():
        rice_merged[cls].extend(paths)

print("All buckets gathered. Checking for issues above (WARNING lines) before proceeding.")

All buckets gathered. Checking for issues above (WARNING lines) before proceeding.


In [6]:
merged_v5 = defaultdict(list)
for bucket in (pv_bucket, pw_bucket, pd_bucket, fp_bucket, rice_merged, wheat_bucket):
    for cls, paths in bucket.items():
        merged_v5[cls].extend(paths)

print(f"Total classes gathered: {len(merged_v5)}")
for cls, paths in sorted(merged_v5.items()):
    print(f"{cls}: {len(paths)} combined")

missing = set(CLASS_NAMES) - set(merged_v5.keys())
if missing:
    print(f"\nMISSING FROM MERGE: {missing}")
else:
    print("\nAll 34 expected classes present.")

Total classes gathered: 34
Maize_Common_Rust: 1582 combined
Maize_Gray_Leaf_Spot: 822 combined
Maize_Healthy: 1391 combined
Maize_Northern_Leaf_Blight: 1391 combined
Potato_Early_Blight: 1384 combined
Potato_Healthy: 401 combined
Potato_Late_Blight: 1440 combined
Rice_Bacterial_Leaf_Blight: 479 combined
Rice_Bacterial_Leaf_Streak: 380 combined
Rice_Bacterial_Panicle_Blight: 337 combined
Rice_Blast: 1936 combined
Rice_Brown_Spot: 1055 combined
Rice_Dead_Heart: 1442 combined
Rice_Downy_Mildew: 620 combined
Rice_Healthy: 1764 combined
Rice_Hispa: 1594 combined
Rice_Leaf_Scald: 143 combined
Rice_Sheath_Blight: 219 combined
Rice_Tungro: 1207 combined
Tomato_Bacterial_Spot: 2508 combined
Tomato_Early_Blight: 1425 combined
Tomato_Healthy: 1874 combined
Tomato_Late_Blight: 2305 combined
Tomato_Leaf_Mold: 1276 combined
Tomato_Mosaic_Virus: 619 combined
Tomato_Septoria_Leaf_Spot: 2136 combined
Tomato_Spider_Mites: 1678 combined
Tomato_Target_Spot: 1404 combined
Tomato_Yellow_Leaf_Curl_Virus: 581

In [7]:
MAX_PER_CLASS = 800
MIN_PER_CLASS = 50
AUGMENT_TARGET = 800

augment_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, shear=10, scale=(0.8, 1.2)),
])

def balance_and_write(merged, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    summary = []
    for cls, paths in merged.items():
        if len(paths) < MIN_PER_CLASS:
            summary.append((cls, len(paths), 'DROPPED'))
            continue
        class_out_dir = os.path.join(output_dir, cls)
        os.makedirs(class_out_dir, exist_ok=True)
        if len(paths) > MAX_PER_CLASS:
            chosen = random.sample(paths, MAX_PER_CLASS)
            for i, src in enumerate(chosen):
                shutil.copy(src, os.path.join(class_out_dir, f"{cls}_{i:05d}.jpg"))
            summary.append((cls, MAX_PER_CLASS, f'downsampled from {len(paths)}'))
        elif len(paths) < AUGMENT_TARGET:
            for i, src in enumerate(paths):
                shutil.copy(src, os.path.join(class_out_dir, f"{cls}_orig_{i:05d}.jpg"))
            n_needed = AUGMENT_TARGET - len(paths)
            i = 0
            while n_needed > 0:
                src = random.choice(paths)
                img = Image.open(src).convert('RGB')
                aug_img = augment_transform(img)
                aug_img.save(os.path.join(class_out_dir, f"{cls}_aug_{i:05d}.jpg"))
                i += 1; n_needed -= 1
            summary.append((cls, AUGMENT_TARGET, f'augmented from {len(paths)}'))
        else:
            for i, src in enumerate(paths):
                shutil.copy(src, os.path.join(class_out_dir, f"{cls}_{i:05d}.jpg"))
            summary.append((cls, len(paths), 'used as-is'))
    print("\n--- Final class summary ---")
    for cls, count, note in sorted(summary):
        print(f"  {cls:35s} {count:5d}  ({note})")
    print(f"\nTotal classes written: {len(os.listdir(output_dir))}")

OUTPUT_DIR_V5 = '/kaggle/working/tier2_unified_v5'
balance_and_write(merged_v5, OUTPUT_DIR_V5)


--- Final class summary ---
  Maize_Common_Rust                     800  (downsampled from 1582)
  Maize_Gray_Leaf_Spot                  800  (downsampled from 822)
  Maize_Healthy                         800  (downsampled from 1391)
  Maize_Northern_Leaf_Blight            800  (downsampled from 1391)
  Potato_Early_Blight                   800  (downsampled from 1384)
  Potato_Healthy                        800  (augmented from 401)
  Potato_Late_Blight                    800  (downsampled from 1440)
  Rice_Bacterial_Leaf_Blight            800  (augmented from 479)
  Rice_Bacterial_Leaf_Streak            800  (augmented from 380)
  Rice_Bacterial_Panicle_Blight         800  (augmented from 337)
  Rice_Blast                            800  (downsampled from 1936)
  Rice_Brown_Spot                       800  (downsampled from 1055)
  Rice_Dead_Heart                       800  (downsampled from 1442)
  Rice_Downy_Mildew                     800  (augmented from 620)
  Rice_Healthy       

In [8]:
TIER2_DATASET = "gulshan07/tier2-unified-dataset"

def push_dataset_folder(folder_path, dataset_id, title, first_time=True):
    meta_path = os.path.join(folder_path, "dataset-metadata.json")
    with open(meta_path, "w") as f:
        json.dump({"title": title, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}, f)
    cmd = ["kaggle", "datasets", "create", "-p", folder_path, "-r", "zip"] if first_time else \
          ["kaggle", "datasets", "version", "-p", folder_path, "-m", "update", "-r", "zip"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print("STDOUT:", result.stdout)
    print("Return code:", result.returncode)

push_dataset_folder(OUTPUT_DIR_V5, TIER2_DATASET, "Tier 2 Unified Dataset (34 classes)", first_time=False)
# first_time=False since gulshan07/tier2-unified-dataset already exists - this updates it

STDOUT: Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.2), please consider upgrading to the latest version (2.2.2)
Starting upload for file Potato_Late_Blight.zip
Upload successful: Potato_Late_Blight.zip (38MB)
Starting upload for file Wheat_Mildew.zip
Upload successful: Wheat_Mildew.zip (255MB)
Starting upload for file Rice_Leaf_Scald.zip
Upload successful: Rice_Leaf_Scald.zip (91MB)
Starting upload for file Maize_Healthy.zip
Upload successful: Maize_Healthy.zip (71MB)
Starting upload for file Tomato_Healthy.zip
Upload successful: Tomato_Healthy.zip (45MB)
Starting upload for file Maize_Gray_Leaf_Spot.zip
Upload successful: Maize_Gray_Leaf_Spot.zip (106MB)
Starting upload for file Potato_Early_Blight.zip
Upload successful: Potato_Early_Blight.zip (40MB)
Starting upload for file Tomato_Septoria_Leaf_Spot.zip
Upload successful: Tomato_Septoria_Leaf_Spot.zip (37MB)
Starting upload for file Rice_Healthy.zip
Upload successful: Rice_Healthy.zip (62MB)
Startin

In [9]:
SAVE_DIR = '/kaggle/working/models'
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH = os.path.join(SAVE_DIR, 'tier2_v5_rice_wheat.pth')

IMG_SIZE = 224
BATCH_SIZE = 64
NUM_EPOCHS = 35
PATIENCE = 7
FREEZE_UNTIL_BLOCK = -4

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset_aug = datasets.ImageFolder(OUTPUT_DIR_V5, transform=train_transform)
full_dataset_plain = datasets.ImageFolder(OUTPUT_DIR_V5, transform=eval_transform)
print(f"Classes ({len(full_dataset_aug.classes)}): {full_dataset_aug.classes}")

targets = np.array(full_dataset_aug.targets)
indices = np.arange(len(targets))
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=targets, random_state=SEED)
train_subset = Subset(full_dataset_aug, train_idx)
val_subset = Subset(full_dataset_plain, val_idx)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

train_labels = targets[train_idx]
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

model = EfficientNet.from_pretrained('efficientnet-b0')
model._fc = nn.Linear(model._fc.in_features, len(full_dataset_aug.classes))
for param in model.parameters():
    param.requires_grad = False
for block in model._blocks[FREEZE_UNTIL_BLOCK:]:
    for param in block.parameters():
        param.requires_grad = True
for param in model._fc.parameters():
    param.requires_grad = True
for param in model._conv_head.parameters():
    param.requires_grad = True
for param in model._bn1.parameters():
    param.requires_grad = True
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

Classes (34): ['Maize_Common_Rust', 'Maize_Gray_Leaf_Spot', 'Maize_Healthy', 'Maize_Northern_Leaf_Blight', 'Potato_Early_Blight', 'Potato_Healthy', 'Potato_Late_Blight', 'Rice_Bacterial_Leaf_Blight', 'Rice_Bacterial_Leaf_Streak', 'Rice_Bacterial_Panicle_Blight', 'Rice_Blast', 'Rice_Brown_Spot', 'Rice_Dead_Heart', 'Rice_Downy_Mildew', 'Rice_Healthy', 'Rice_Hispa', 'Rice_Leaf_Scald', 'Rice_Sheath_Blight', 'Rice_Tungro', 'Tomato_Bacterial_Spot', 'Tomato_Early_Blight', 'Tomato_Healthy', 'Tomato_Late_Blight', 'Tomato_Leaf_Mold', 'Tomato_Mosaic_Virus', 'Tomato_Septoria_Leaf_Spot', 'Tomato_Spider_Mites', 'Tomato_Target_Spot', 'Tomato_Yellow_Leaf_Curl_Virus', 'Wheat_Brown_Rust', 'Wheat_Healthy', 'Wheat_Mildew', 'Wheat_Septoria', 'Wheat_Yellow_Rust']
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth



100%|██████████| 20.4M/20.4M [00:00<00:00, 223MB/s]

Loaded pretrained weights for efficientnet-b0


In [10]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return running_loss / len(loader.dataset), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return running_loss / len(loader.dataset), correct / total, macro_f1, all_preds, all_labels

In [11]:
best_f1 = 0.0
epochs_no_improve = 0
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} Macro-F1: {val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1 = val_f1; epochs_no_improve = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  -> saved (Val Macro-F1: {val_f1:.4f})")
        display(FileLink(SAVE_PATH))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

Epoch 1/35 | Train Loss: 1.8853 Acc: 0.5087 | Val Loss: 0.9321 Acc: 0.7048 Macro-F1: 0.7058
  -> saved (Val Macro-F1: 0.7058)


/kaggle/working/models/tier2_v5_rice_wheat.pth

Epoch 2/35 | Train Loss: 0.8641 Acc: 0.7275 | Val Loss: 0.6110 Acc: 0.7932 Macro-F1: 0.7931
  -> saved (Val Macro-F1: 0.7931)


/kaggle/working/models/tier2_v5_rice_wheat.pth

Epoch 3/35 | Train Loss: 0.6277 Acc: 0.7993 | Val Loss: 0.4774 Acc: 0.8381 Macro-F1: 0.8382
  -> saved (Val Macro-F1: 0.8382)


...

Epoch 25/35 | Train Loss: 0.1003 Acc: 0.9684 | Val Loss: 0.2501 Acc: 0.9305 Macro-F1: 0.9307
Epoch 26/35 | Train Loss: 0.0982 Acc: 0.9685 | Val Loss: 0.2475 Acc: 0.9309 Macro-F1: 0.9309
  -> saved (Val Macro-F1: 0.9309)


/kaggle/working/models/tier2_v5_rice_wheat.pth

Epoch 27/35 | Train Loss: 0.0935 Acc: 0.9705 | Val Loss: 0.2446 Acc: 0.9300 Macro-F1: 0.9301
Epoch 28/35 | Train Loss: 0.0916 Acc: 0.9709 | Val Loss: 0.2459 Acc: 0.9314 Macro-F1: 0.9315
  -> saved (Val Macro-F1: 0.9315)


/kaggle/working/models/tier2_v5_rice_wheat.pth

Epoch 29/35 | Train Loss: 0.0952 Acc: 0.9701 | Val Loss: 0.2506 Acc: 0.9307 Macro-F1: 0.9308
Epoch 30/35 | Train Loss: 0.0898 Acc: 0.9723 | Val Loss: 0.2487 Acc: 0.9303 Macro-F1: 0.9304
Epoch 31/35 | Train Loss: 0.0928 Acc: 0.9700 | Val Loss: 0.2483 Acc: 0.9301 Macro-F1: 0.9302
Epoch 32/35 | Train Loss: 0.0912 Acc: 0.9708 | Val Loss: 0.2497 Acc: 0.9296 Macro-F1: 0.9296
Epoch 33/35 | Train Loss: 0.0883 Acc: 0.9722 | Val Loss: 0.2491 Acc: 0.9301 Macro-F1: 0.9302
Epoch 34/35 | Train Loss: 0.0870 Acc: 0.9720 | Val Loss: 0.2493 Acc: 0.9305 Macro-F1: 0.9306
Epoch 35/35 | Train Loss: 0.0878 Acc: 0.9729 | Val Loss: 0.2494 Acc: 0.9307 Macro-F1: 0.9307
Early stopping at epoch 35


In [12]:
model.load_state_dict(torch.load(SAVE_PATH, weights_only=True))
_, _, _, all_preds, all_labels = evaluate(model, val_loader, criterion, device)
print(classification_report(all_labels, all_preds, target_names=full_dataset_aug.classes))

                               precision    recall  f1-score   support

            Maize_Common_Rust       0.94      0.95      0.95       160
         Maize_Gray_Leaf_Spot       0.84      0.83      0.84       160
                Maize_Healthy       0.99      0.96      0.97       160
   Maize_Northern_Leaf_Blight       0.83      0.85      0.84       160
          Potato_Early_Blight       0.86      0.91      0.88       160
               Potato_Healthy       0.91      1.00      0.95       160
           Potato_Late_Blight       0.91      0.84      0.87       160
   Rice_Bacterial_Leaf_Blight       0.95      0.97      0.96       160
   Rice_Bacterial_Leaf_Streak       0.95      0.99      0.97       160
Rice_Bacterial_Panicle_Blight       0.97      0.97      0.97       160
                   Rice_Blast       0.91      0.88      0.90       160
              Rice_Brown_Spot       0.97      0.91      0.94       160
              Rice_Dead_Heart       0.98      0.96      0.97       160
     

In [13]:
class PlantDocTestOnlyDataset(Dataset):
    def __init__(self, plantdoc_root, class_map, class_names, transform):
        self.samples = []
        self.transform = transform
        test_dir = os.path.join(plantdoc_root, 'test')
        for source_class, unified_class in class_map.items():
            class_dir = os.path.join(test_dir, source_class)
            if not os.path.isdir(class_dir):
                print(f"  WARNING: not found: {class_dir}")
                continue
            label_idx = class_names.index(unified_class)
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(class_dir, fname), label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        return self.transform(image), label

plantdoc_test_only = PlantDocTestOnlyDataset(plantdoc_dataset_root, PLANTDOC_CLASS_MAP, CLASS_NAMES, eval_transform)
print(f"Total test-only PlantDoc images: {len(plantdoc_test_only)}")
plantdoc_test_loader = DataLoader(plantdoc_test_only, batch_size=32, shuffle=False, num_workers=2)

model.load_state_dict(torch.load(SAVE_PATH, weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in plantdoc_test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(f"Collected {len(all_labels)} predictions")

present_labels = sorted(set(all_labels))
present_names = [CLASS_NAMES[i] for i in present_labels]

print("--- PlantDoc TEST-ONLY evaluation (34-class Tier 2 model, on the 17 overlapping classes) ---")
print(classification_report(all_labels, all_preds, labels=present_labels, target_names=present_names))

Total test-only PlantDoc images: 127
Collected 127 predictions
--- PlantDoc TEST-ONLY evaluation (34-class Tier 2 model, on the 17 overlapping classes) ---
                               precision    recall  f1-score   support

            Maize_Common_Rust       0.90      0.90      0.90        10
         Maize_Gray_Leaf_Spot       0.18      0.50      0.27         4
   Maize_Northern_Leaf_Blight       0.71      0.42      0.53        12
          Potato_Early_Blight       0.55      0.43      0.48        14
           Potato_Late_Blight       0.40      0.75      0.52         8
        Tomato_Bacterial_Spot       0.33      0.22      0.27         9
          Tomato_Early_Blight       0.47      0.78      0.58         9
               Tomato_Healthy       0.40      0.25      0.31         8
           Tomato_Late_Blight       1.00      0.20      0.33        10
             Tomato_Leaf_Mold       0.33      0.67      0.44         6
          Tomato_Mosaic_Virus       0.45      0.50      0.48  

# stoping point

In [4]:
import os, random, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from efficientnet_pytorch import EfficientNet
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
from PIL import Image
import kagglehub

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------------------------------------------------------------------------
# Original, raw PlantVillage - all 38 classes, no pre-augmentation
# ---------------------------------------------------------------------------
pv_path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
PLANTVILLAGE_DIR = os.path.join(pv_path, 'color')
all_classes = sorted(os.listdir(PLANTVILLAGE_DIR))
print(f"Found {len(all_classes)} classes")

# ---------------------------------------------------------------------------
# Gather raw counts per class
# ---------------------------------------------------------------------------
from collections import defaultdict
merged = defaultdict(list)
for cls in all_classes:
    class_dir = os.path.join(PLANTVILLAGE_DIR, cls)
    for fname in os.listdir(class_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            merged[cls].append(os.path.join(class_dir, fname))

for cls, paths in sorted(merged.items()):
    print(f"{cls}: {len(paths)} raw images")

# ---------------------------------------------------------------------------
# Balance - same rules as before: cap large, augment small, drop tiny
# ---------------------------------------------------------------------------
MAX_PER_CLASS = 800
MIN_PER_CLASS = 50
AUGMENT_TARGET = 800

augment_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, shear=10, scale=(0.8, 1.2)),
])

def balance_and_write(merged, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    summary = []
    for cls, paths in merged.items():
        if len(paths) < MIN_PER_CLASS:
            summary.append((cls, len(paths), 'DROPPED'))
            continue
        class_out_dir = os.path.join(output_dir, cls)
        os.makedirs(class_out_dir, exist_ok=True)
        if len(paths) > MAX_PER_CLASS:
            chosen = random.sample(paths, MAX_PER_CLASS)
            for i, src in enumerate(chosen):
                shutil.copy(src, os.path.join(class_out_dir, f"{cls}_{i:05d}.jpg"))
            summary.append((cls, MAX_PER_CLASS, f'downsampled from {len(paths)}'))
        elif len(paths) < AUGMENT_TARGET:
            for i, src in enumerate(paths):
                shutil.copy(src, os.path.join(class_out_dir, f"{cls}_orig_{i:05d}.jpg"))
            n_needed = AUGMENT_TARGET - len(paths)
            i = 0
            while n_needed > 0:
                src = random.choice(paths)
                img = Image.open(src).convert('RGB')
                aug_img = augment_transform(img)
                aug_img.save(os.path.join(class_out_dir, f"{cls}_aug_{i:05d}.jpg"))
                i += 1; n_needed -= 1
            summary.append((cls, AUGMENT_TARGET, f'augmented from {len(paths)}'))
        else:
            for i, src in enumerate(paths):
                shutil.copy(src, os.path.join(class_out_dir, f"{cls}_{i:05d}.jpg"))
            summary.append((cls, len(paths), 'used as-is'))
    print("\n--- Final class summary ---")
    for cls, count, note in sorted(summary):
        print(f"  {cls:45s} {count:5d}  ({note})")

OUTPUT_DIR_ALLPLANT = '/kaggle/working/allplant_balanced'
balance_and_write(merged, OUTPUT_DIR_ALLPLANT)

Device: cuda
Found 38 classes
Apple___Apple_scab: 630 raw images
Apple___Black_rot: 621 raw images
Apple___Cedar_apple_rust: 275 raw images
Apple___healthy: 1645 raw images
Blueberry___healthy: 1502 raw images
Cherry_(including_sour)___Powdery_mildew: 1052 raw images
Cherry_(including_sour)___healthy: 854 raw images
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 513 raw images
Corn_(maize)___Common_rust_: 1192 raw images
Corn_(maize)___Northern_Leaf_Blight: 985 raw images
Corn_(maize)___healthy: 1162 raw images
Grape___Black_rot: 1180 raw images
Grape___Esca_(Black_Measles): 1383 raw images
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 1076 raw images
Grape___healthy: 423 raw images
Orange___Haunglongbing_(Citrus_greening): 5507 raw images
Peach___Bacterial_spot: 2297 raw images
Peach___healthy: 360 raw images
Pepper,_bell___Bacterial_spot: 997 raw images
Pepper,_bell___healthy: 1478 raw images
Potato___Early_blight: 1000 raw images
Potato___Late_blight: 1000 raw images
Potato___hea

In [5]:
SAVE_DIR = '/kaggle/working/models'
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH = os.path.join(SAVE_DIR, 'allplant_38class_baseline.pth')

IMG_SIZE = 224
BATCH_SIZE = 64
NUM_EPOCHS = 30
PATIENCE = 6
FREEZE_UNTIL_BLOCK = -4

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset_aug = datasets.ImageFolder(OUTPUT_DIR_ALLPLANT, transform=train_transform)
full_dataset_plain = datasets.ImageFolder(OUTPUT_DIR_ALLPLANT, transform=eval_transform)
print(f"Classes ({len(full_dataset_aug.classes)})")

targets = np.array(full_dataset_aug.targets)
indices = np.arange(len(targets))
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=targets, random_state=SEED)
train_subset = Subset(full_dataset_aug, train_idx)
val_subset = Subset(full_dataset_plain, val_idx)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

train_labels = targets[train_idx]
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

model = EfficientNet.from_pretrained('efficientnet-b0')
model._fc = nn.Linear(model._fc.in_features, len(full_dataset_aug.classes))
for param in model.parameters():
    param.requires_grad = False
for block in model._blocks[FREEZE_UNTIL_BLOCK:]:
    for param in block.parameters():
        param.requires_grad = True
for param in model._fc.parameters():
    param.requires_grad = True
for param in model._conv_head.parameters():
    param.requires_grad = True
for param in model._bn1.parameters():
    param.requires_grad = True
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return running_loss / len(loader.dataset), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return running_loss / len(loader.dataset), correct / total, macro_f1, all_preds, all_labels

best_f1 = 0.0
epochs_no_improve = 0
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} Macro-F1: {val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1 = val_f1; epochs_no_improve = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  -> saved (Val Macro-F1: {val_f1:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(SAVE_PATH, weights_only=True))
_, _, _, all_preds, all_labels = evaluate(model, val_loader, criterion, device)
print(classification_report(all_labels, all_preds, target_names=full_dataset_aug.classes))

Classes (38)
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 152MB/s]


Loaded pretrained weights for efficientnet-b0
Epoch 1/30 | Train Loss: 1.3296 Acc: 0.7592 | Val Loss: 0.2500 Acc: 0.9428 Macro-F1: 0.9420
  -> saved (Val Macro-F1: 0.9420)
Epoch 2/30 | Train Loss: 0.2106 Acc: 0.9463 | Val Loss: 0.1121 Acc: 0.9689 Macro-F1: 0.9687
  -> saved (Val Macro-F1: 0.9687)
Epoch 3/30 | Train Loss: 0.1146 Acc: 0.9676 | Val Loss: 0.0793 Acc: 0.9742 Macro-F1: 0.9740
  -> saved (Val Macro-F1: 0.9740)
Epoch 4/30 | Train Loss: 0.0794 Acc: 0.9788 | Val Loss: 0.0665 Acc: 0.9786 Macro-F1: 0.9784
  -> saved (Val Macro-F1: 0.9784)
Epoch 5/30 | Train Loss: 0.0629 Acc: 0.9822 | Val Loss: 0.0495 Acc: 0.9834 Macro-F1: 0.9834
  -> saved (Val Macro-F1: 0.9834)
Epoch 6/30 | Train Loss: 0.0504 Acc: 0.9847 | Val Loss: 0.0436 Acc: 0.9870 Macro-F1: 0.9870
  -> saved (Val Macro-F1: 0.9870)
Epoch 7/30 | Train Loss: 0.0431 Acc: 0.9878 | Val Loss: 0.0398 Acc: 0.9860 Macro-F1: 0.9860
Epoch 8/30 | Train Loss: 0.0414 Acc: 0.9872 | Val Loss: 0.0356 Acc: 0.9898 Macro-F1: 0.9898
  -> saved (Va

In [6]:
from IPython.display import FileLink
FileLink('/kaggle/working/models/allplant_38class_baseline.pth')

/kaggle/working/models/allplant_38class_baseline.pth